# Chapter 18: Neural Networks. Computing exercises

These exercises are from *Bayes, Artificial Intelligence, and Deep Learning: Foundations of Data Science* by Nick Polson and Vadim Sokolov, <https://github.com/VadimSokolov/bayes_ai_deep_learning>. The text is licensed under CC BY 4.0 and the code under the MIT License.

The code of the chapter, which several exercises refer to, is in [`code/18-nn.R`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/18-nn.R) and [`code/18-nn.py`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/18-nn.py).

This notebook holds the exercises of the chapter that use Python; the ones that use R are in [`18-nn-R.ipynb`](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/18-nn-R.ipynb).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/18-nn-Python.ipynb)

In [ ]:
# Modules that the exercises of this notebook use: matplotlib, numpy, torch (all installed on Colab)

## Exercise 18.9: Autoencoder Bottleneck versus PCA

The chapter's auto-encoder equation reconstructs an $M$-dimensional input through a $K$-dimensional bottleneck ($K<M$) as $Y_j(x) = \sum_{k=1}^K W_2^{jk} f\big(\sum_{i=1}^M W_1^{ki} x_i\big)$, with biases set to zero. Here you take $f$ to be the identity (a *linear* autoencoder) and compare it directly to Principal Component Analysis.

In [ ]:
import numpy as np

np.random.seed(0)
M, K, N = 6, 2, 300                                # ambient dimension, true rank, sample size
B, _ = np.linalg.qr(np.random.randn(M, K))         # M x K orthonormal loading directions
Z = np.random.randn(N, K) @ np.diag([3.0, 1.0])    # latent factors on unequal scales
X = Z @ B.T + 0.05 * np.random.randn(N, M)         # M-dim data near a K-dim subspace
X = X - X.mean(axis=0)

`X` has $N=300$ rows (samples) and $M=6$ columns (features), the transpose of the chapter's $X \in \mathbb{R}^{M\times N}$ convention; it was generated from a $K=2$-dimensional linear subspace plus small isotropic noise.

a) Using the SVD of the (already centered) `X`, compute the top-$K$ principal directions $V \in \mathbb{R}^{M\times K}$ and the rank-$K$ PCA reconstruction $XVV^\top$. Report the reconstruction MSE (mean squared error over all $N\times M$ entries) and the fraction of variance explained by the top two singular values.

b) Build a linear autoencoder in PyTorch: `encoder = nn.Linear(M, K, bias=False)` followed by `decoder = nn.Linear(K, M, bias=False)`, with no nonlinearity between them, so the fitted values are $\hat X = XW_1^\top W_2^\top$ with $W_1$ ($K\times M$) the encoder weight and $W_2$ ($M\times K$) the decoder weight (matching the chapter's $W_1, W_2$ with $f=\text{identity}$ and $\lambda=0$, i.e., no ridge penalty). Train by minimizing the mean squared reconstruction error with Adam. Report the final reconstruction MSE and compare it to (a).

c) The autoencoder's $K$-dimensional output subspace is the column space of the decoder weight $W_2$. Compute an orthonormal basis $Q$ for this column space (e.g., via `np.linalg.qr(W2)`), then compute the principal angles between this subspace and the PCA subspace spanned by $V$ from (a), using the singular values of $Q^\top V$ (these singular values are the cosines of the principal angles). Report the angles in degrees. What do they show?

d) The autoencoder's loss places no orthogonality constraint on $W_1$ and $W_2$: for any invertible $K\times K$ matrix $R$, replacing $W_1 \to RW_1$ and $W_2 \to W_2R^{-1}$ leaves $W_2W_1$, and hence every fitted value, unchanged. Given this "gauge freedom" and your results in (b)-(c), explain why the autoencoder nonetheless recovers the same *subspace* as PCA's top-$K$ components, even though its individual weight vectors need not match the PCA loadings themselves.

## Exercise 18.10: Variance Reduction in Correlated Ensembles

The chapter argues that averaging $M$ models "with the same bias but independent estimation errors" reduces the variance term "by a factor of $M$, assuming the models are uncorrelated," and then cautions that "in practice, models are not perfectly independent, but ensembles still reduce variance" (Dietterich 2000). This exercise makes the correlated case precise and demonstrates it with an ensemble of neural networks.

Consider the chapter's ensemble average $\hat Y_{\text{ens}}(X) = \frac{1}{M}\sum_{m=1}^M f_m(X)$ of $M$ regression models predicting $Y = f(X) + \epsilon$ with $\epsilon \sim N(0, \sigma_\epsilon^2)$. Assume every model has the same conditional bias $\mathrm{E}\left(f_m(X)\right) - f(X) = \beta$, the same conditional variance $\mathrm{Var}\left(f_m(X)\right) = \sigma^2$, and the same pairwise correlation $\rho = \mathrm{Cor}\left(f_m(X), f_{m'}(X)\right)$ for all $m \neq m'$.

a) Show that averaging leaves the bias unchanged, $\mathrm{E}\left(\hat Y_{\text{ens}}(X)\right) - f(X) = \beta$, and that
$$
\mathrm{Var}\left(\hat Y_{\text{ens}}(X)\right) = \rho\,\sigma^2 + \frac{1-\rho}{M}\,\sigma^2 .
$$
Verify that $\rho = 0$ recovers the chapter's factor-of-$M$ reduction, and identify the limiting variance as $M \to \infty$.

b) The expected test error at $X$ decomposes as $\text{Bias}^2 + \mathrm{Var}\left(\hat Y_{\text{ens}}(X)\right) + \sigma_\epsilon^2$. Using $\beta = 0.4$, $\sigma^2 = 2.0$, and $\sigma_\epsilon^2 = 0.5$, compute this error for (i) a single model ($M = 1$); (ii) $M = 5$ uncorrelated models ($\rho = 0$); (iii) $M = 5$ correlated models ($\rho = 0.3$). Then find the smallest integer $M$ for which the correlated ensemble ($\rho = 0.3$) reaches a variance within $10\%$ of its $M \to \infty$ floor.

c) Train an ensemble of $M = 10$ small networks (identical architecture, different random seeds) on a noisy regression target, repeating over many fresh datasets so the members share a common data-driven error component. Estimate the per-model error variance $\sigma^2$ and the average pairwise error correlation $\rho$, and measure the ensemble error variance as a function of $M$. Plot the measured variance together with the formula from (a), the uncorrelated prediction $\sigma^2/M$, and the floor $\rho\sigma^2$.

d) Use the estimated $\rho$ and $\sigma^2$ to explain the chapter's remark that ensembles "still reduce variance" despite correlation. Why does the measured variance plateau instead of falling to zero, and what does this imply about the value of *decorrelating* members (snapshot ensembles, multi-architecture ensembles, dropout) rather than merely adding more of them?